# 02 - Statistical analysis
Reads `analysis_df.csv` from `01-merge-and-prep`. Needs `statsmodels` (`python -m uv add statsmodels`).
Unit of analysis is the comment. Standard errors are clustered on `video_id` when it exists.
Caveat: codes come from prompt v1.4 on a 4-bit model, so all results describe the LLM coding, pending the hand-coded check at the end.

In [23]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

# Repo root: the nearest folder (cwd or a parent) that holds 03-llm-passes/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03-llm-passes").is_dir())
os.chdir(ROOT)
CRISIS = "ateneo-drowning"

# Define both the base output folder and the crisis-specific folder
BASE_OUT = Path("04-statistical-analysis/output")
OUT = BASE_OUT / CRISIS

PHASES = ["01-onset", "02-confirmation", "03-spread", "04-reckoning", "05-lingering"]
EMO = ["NEG_EMOTION", "ANY_EMOTION", "BROAD_OUTRAGE", "ANGER", "CONTEMPT", "DISGUST",
       "SADNESS", "CONDOLENCE", "COMPASSION_VICTIMS", "SYMPATHY_OFFENDER", "FEAR"]

# Written by 01-merge-and-prep.ipynb
df = pd.read_csv(OUT / "analysis_df.csv")

df["phase"] = pd.Categorical(df.phase, PHASES, ordered=True)
top = df.LANGUAGE.value_counts().index[:3]
df["lang3"] = df.LANGUAGE.where(df.LANGUAGE.isin(top), "OTHER")
group_col = df["video_id"] if "video_id" in df else df["comment_id"]
df["cluster"] = group_col.factorize()[0]
print("clustering on:", "video_id" if "video_id" in df else "comment_id (no clustering)")
df.shape

clustering on: video_id


(11551, 95)

In [24]:
def fit(formula, d):
    return smf.logit(formula, d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.cluster})

def or_row(m, term):
    ci = m.conf_int().loc[term]
    return pd.Series({"OR": np.exp(m.params[term]), "lo": np.exp(ci[0]),
                      "hi": np.exp(ci[1]), "p": m.pvalues[term]})

def ca_trend(x, n, s):
    x, n, s = map(np.asarray, (x, n, s))
    N = n.sum()
    p = x.sum() / N
    t = (s * (x - n * p)).sum()
    v = p * (1 - p) * ((n * s**2).sum() - (n * s).sum()**2 / N)
    z = t / np.sqrt(v)
    return z, 2 * stats.norm.sf(abs(z))

## RQ1 - How much class talk, and does it change across phases?
To determine the baseline prevalence of class-framed discourse and evaluate whether its volume fluctuates across the five crisis phases.

**Statistical Methods:** 
* **Wilson Score Intervals:** To calculate the 95% confidence intervals for the proportion of class talk in each phase.
* **Chi-Square Test of Independence:** To test for an overall relationship between crisis phase and the presence of a class frame[cite: 2].
* **Cochran-Armitage Trend Test:** To determine if there is a linear directional trend (increasing or decreasing) in class talk over time[cite: 2].

**Hypotheses:**
* **Null Hypothesis (H0):** The proportion of class-framed comments remains stable and independent of the crisis phase.
* **Alternative Hypothesis (Ha):** The proportion of class-framed comments changes significantly, showing a distinct trend as the crisis evolves.

In [25]:
g = df.groupby("phase", observed=True).CLASS_FRAME.agg(["sum", "count"])
lo, hi = proportion_confint(g["sum"], g["count"], method="wilson")
g["share"] = g["sum"] / g["count"]
g["ci_lo"], g["ci_hi"] = lo, hi
g.round(3)

,sum,count,share,ci_lo,ci_hi
phase,,,,,
01-onset,30,855,0.035,0.025,0.050
02-confirmation,110,2649,0.042,0.035,0.050
03-spread,180,5375,0.033,0.029,0.039
04-reckoning,48,1410,0.034,0.026,0.045
05-lingering,46,1262,0.036,0.027,0.048


In [ ]:
# 1. Reset the index so 'phase' becomes a standard column
rq1_table = g.reset_index().copy()

# 2. Combine the 'ci_lo' and 'ci_hi' columns into a single formatted 95% CI string
rq1_table["95% CI (ci_lo, ci_hi)"] = "[" + rq1_table["ci_lo"].round(3).astype(str) + ", " + rq1_table["ci_hi"].round(3).astype(str) + "]"

# 3. Format the share as a percentage string for easier reading in a paper (optional)
rq1_table["Class Talk Share"] = (rq1_table["share"] * 100).round(1).astype(str) + "%"

# 4. Rename the existing columns to match your academic headers
rq1_table = rq1_table.rename(columns={
    "phase": "Crisis Phase",
    "sum": "Class Talk Volume",
    "count": "Total Comments"
})

# 5. Select only the specific columns you want to show in the final table
final_rq1_table = rq1_table[[
    "Crisis Phase", 
    "Total Comments", 
    "Class Talk Volume", 
    "Class Talk Share", 
    "95% CI (ci_lo, ci_hi)"
]]

# 6. Display the cleanly formatted table
display(final_rq1_table)

# Optional: Export to HTML to copy-paste into your manuscript
final_rq1_table.to_html("rq1_volume_table.html", index=False)

In [11]:
tab = pd.crosstab(df.phase, df.CLASS_FRAME)
chi2, p, dof, exp = stats.chi2_contingency(tab)
print(f"chi2({dof}) = {chi2:.2f}, p = {p:.3f}, Cramer's V = {np.sqrt(chi2 / tab.values.sum()):.3f}")

z, pz = ca_trend(g["sum"], g["count"], np.arange(len(g)))
print(f"Cochran-Armitage trend: z = {z:.2f}, p = {pz:.3f}")

rp, cp = tab.sum(1).values[:, None] / tab.values.sum(), tab.sum(0).values / tab.values.sum()
adj = (tab.values - exp) / np.sqrt(exp * (1 - rp) * (1 - cp))
print(pd.DataFrame(adj, index=tab.index, columns=tab.columns).round(2))

m = fit("CLASS_FRAME ~ phase_num + log_len + C(lang3)", df)
print(or_row(m, "phase_num").round(3))

chi2(4) = 3.50, p = 0.478, Cramer's V = 0.017
Cochran-Armitage trend: z = -0.71, p = 0.478
CLASS_FRAME         0     1
phase                      
01-onset         0.12 -0.12
02-confirmation -1.79  1.79
03-spread        1.27 -1.27
04-reckoning     0.39 -0.39
05-lingering    -0.12  0.12


C:\Users\PHSP8131\AppData\Local\Temp\ipykernel_10140\2214095534.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  rp, cp = tab.sum(1).values[:, None] / tab.values.sum(), tab.sum(0).values / tab.values.sum()
C:\Users\PHSP8131\AppData\Local\Temp\ipykernel_10140\2214095534.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  rp, cp = tab.sum(1).values[:, None] / tab.values.sum(), tab.sum(0).values / tab.values.sum()


OR    0.973
lo    0.806
hi    1.175
p     0.776
dtype: float64


c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


### Direction of class talk (class-framed comments only)

In [12]:
cf = df[df.CLASS_FRAME == 1].copy()
cf["dir3"] = cf.CLASS_DIRECTION.replace({"INSTITUTION_ELITE": "ELITE_OR_OTHER", "OTHER": "ELITE_OR_OTHER"})
print(cf.dir3.value_counts(), "\n")
print(pd.crosstab(cf.phase, cf.dir3))
print(pd.crosstab(cf.phase, cf.dir3, normalize="index").round(2))

dir3
INEQUALITY_JUSTICE    215
VICTIMS_POOR          187
ELITE_OR_OTHER         12
Name: count, dtype: int64 

dir3             ELITE_OR_OTHER  INEQUALITY_JUSTICE  VICTIMS_POOR
phase                                                            
01-onset                      0                  11            19
02-confirmation               5                  60            45
03-spread                     6                  94            80
04-reckoning                  1                  25            22
05-lingering                  0                  25            21
dir3             ELITE_OR_OTHER  INEQUALITY_JUSTICE  VICTIMS_POOR
phase                                                            
01-onset                   0.00                0.37          0.63
02-confirmation            0.05                0.55          0.41
03-spread                  0.03                0.52          0.44
04-reckoning               0.02                0.52          0.46
05-lingering               0.00

In [13]:
def perm_chi2(a, b, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = pd.factorize(a)[0], pd.factorize(b)[0]
    ka, kb = a.max() + 1, b.max() + 1
    def stat(bb):
        t = np.zeros((ka, kb))
        np.add.at(t, (a, bb), 1)
        return stats.chi2_contingency(t, correction=False)[0]
    obs = stat(b)
    sims = np.array([stat(rng.permutation(b)) for _ in range(n)])
    return obs, (1 + (sims >= obs).sum()) / (n + 1)

obs, p = perm_chi2(cf.phase.astype(str), cf.dir3)
print(f"phase x direction: chi2 = {obs:.2f}, permutation p = {p:.3f}")

phase x direction: chi2 = 7.67, permutation p = 0.473


## RQ2 & RQ3: The Emotion Profile of Class-Framed Discourse
**Objective:** To determine how class talk functions socially by comparing its emotional signature to standard crisis communication. 

**Statistical Methods:**
* **Logistic Regression:** Used to calculate the Odds Ratio (OR) for specific emotions occurring in class-framed comments, controlling for comment length (`log_len`) and language (`C(lang3)`)[cite: 3].
* **Holm-Bonferroni Method:** Applied to correct p-values for multiple comparisons across 11 distinct emotional outcomes, minimizing false positives.

**Hypotheses:**
* **Null Hypothesis (H0):** Class-framed comments are just as likely to contain specific emotions (e.g., sadness, anger) as non-class-framed comments.
* **Alternative Hypothesis (Ha):** Class-framed comments possess a distinctly different emotional profile compared to standard comments.

In [14]:
def emotion_table(d):
    rows = []
    for y in EMO:
        r = {"outcome": y,
             "rate_class": d.loc[d.CLASS_FRAME == 1, y].mean(),
             "rate_other": d.loc[d.CLASS_FRAME == 0, y].mean()}
        try:
            m = fit(f"{y} ~ CLASS_FRAME + C(phase) + C(lang3) + log_len", d)
            r.update(or_row(m, "CLASS_FRAME"))
        except Exception as e:
            r["note"] = type(e).__name__
        rows.append(r)
    t = pd.DataFrame(rows).set_index("outcome")
    ok = t.p.notna()
    t.loc[ok, "p_holm"] = multipletests(t.loc[ok, "p"], method="holm")[1]
    return t

emo_tab = emotion_table(df)
emo_tab.round(3)

c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\di

,rate_class,rate_other,OR,lo,hi,p,p_holm
outcome,,,,,,,
NEG_EMOTION,0.326,0.354,0.567,0.384,0.836,0.004,0.021
ANY_EMOTION,0.444,0.562,0.325,0.254,0.417,0.000,0.000
BROAD_OUTRAGE,0.167,0.169,0.676,0.490,0.933,0.017,0.069
ANGER,0.312,0.319,0.633,0.420,0.954,0.029,0.087
CONTEMPT,0.060,0.101,0.410,0.297,0.567,0.000,0.000
DISGUST,0.010,0.004,1.807,0.688,4.750,0.230,0.415
SADNESS,0.128,0.201,0.379,0.291,0.493,0.000,0.000
CONDOLENCE,0.060,0.134,0.309,0.182,0.524,0.000,0.000
COMPASSION_VICTIMS,0.029,0.093,0.164,0.106,0.255,0.000,0.000


In [15]:
emo_tab.to_csv(OUT / "emotion_by_class.csv")

Summary table

In [16]:
# 1. Create a copy and reset the index so 'outcome' becomes a standard column
clean_table = emo_tab.reset_index().copy()

# 2. Combine the 'lo' and 'hi' columns into a single formatted 95% CI string column
clean_table["95% CI (lo, hi)"] = "[" + clean_table["lo"].round(3).astype(str) + ", " + clean_table["hi"].round(3).astype(str) + "]"

# 3. Rename the existing columns to match your academic headers
clean_table = clean_table.rename(columns={
    "outcome": "Emotional Outcome",
    "rate_class": "Class Talk Rate",
    "rate_other": "Other Comments Rate",
    "OR": "Odds Ratio (OR)",
    "p_holm": "Adj. p-value (Holm)"
})

# 4. Select only the specific columns you want to show in the final table
final_table = clean_table[[
    "Emotional Outcome", 
    "Class Talk Rate", 
    "Other Comments Rate", 
    "Odds Ratio (OR)", 
    "95% CI (lo, hi)", 
    "Adj. p-value (Holm)"
]].round(3) # Round the remaining numeric columns to 3 decimal places

# 5. Display the beautifully formatted table
display(final_table)

,Emotional Outcome,Class Talk Rate,Other Comments Rate,Odds Ratio (OR),"95% CI (lo, hi)",Adj. p-value (Holm)
0,NEG_EMOTION,0.326,0.354,0.567,"[0.384, 0.836]",0.021
1,ANY_EMOTION,0.444,0.562,0.325,"[0.254, 0.417]",0.000
2,BROAD_OUTRAGE,0.167,0.169,0.676,"[0.49, 0.933]",0.069
3,ANGER,0.312,0.319,0.633,"[0.42, 0.954]",0.087
4,CONTEMPT,0.060,0.101,0.410,"[0.297, 0.567]",0.000
5,DISGUST,0.010,0.004,1.807,"[0.688, 4.75]",0.415
6,SADNESS,0.128,0.201,0.379,"[0.291, 0.493]",0.000
7,CONDOLENCE,0.060,0.134,0.309,"[0.182, 0.524]",0.000
8,COMPASSION_VICTIMS,0.029,0.093,0.164,"[0.106, 0.255]",0.000
9,SYMPATHY_OFFENDER,0.000,0.002,0.000,"[0.0, 0.0]",0.000


### Within class talk: does the direction of class talk carry different emotions?

In [17]:
two = cf[cf.dir3.isin(["INEQUALITY_JUSTICE", "VICTIMS_POOR"])]
rows = []
for y in EMO:
    a = two.loc[two.dir3 == "INEQUALITY_JUSTICE", y]
    b = two.loc[two.dir3 == "VICTIMS_POOR", y]
    odds, p = stats.fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
    rows.append({"outcome": y, "rate_ineq_justice": a.mean(), "rate_victims_poor": b.mean(), "OR": odds, "p": p})
within = pd.DataFrame(rows).set_index("outcome")
within["p_holm"] = multipletests(within.p, method="holm")[1]
within.round(3)

,rate_ineq_justice,rate_victims_poor,OR,p,p_holm
outcome,,,,,
NEG_EMOTION,0.372,0.283,1.498,0.071,0.708
ANY_EMOTION,0.447,0.460,0.947,0.841,1.000
BROAD_OUTRAGE,0.158,0.176,0.877,0.688,1.000
ANGER,0.349,0.278,1.391,0.134,1.000
CONTEMPT,0.070,0.053,1.328,0.541,1.000
DISGUST,0.014,0.005,2.632,0.627,1.000
SADNESS,0.107,0.160,0.627,0.139,1.000
CONDOLENCE,0.037,0.091,0.386,0.037,0.405
COMPASSION_VICTIMS,0.023,0.037,0.612,0.559,1.000


## Phase and Class Interaction
**Objective:** To test the "emotional turn" argument by evaluating if the specific emotions attached to class talk shift as the crisis matures.

**Statistical Methods:**
* **Likelihood-Ratio (LR) Test:** Compares a baseline logistic model against an interaction model (`CLASS_FRAME * phase_num`) to see if the passage of time significantly alters the emotional weight of class talk.

**Hypotheses:**
* **Null Hypothesis (H0):** The emotional profile of class talk is static; the crisis phase does not interact with the class frame.
* **Alternative Hypothesis (Ha):** There is a significant interaction effect between class framing and phase, meaning the emotions embedded in class talk evolve over the crisis lifecycle.

In [18]:
rows = []
for y in ["NEG_EMOTION", "ANGER", "CONTEMPT", "SADNESS", "COMPASSION_VICTIMS", "BROAD_OUTRAGE"]:
    m0 = smf.logit(f"{y} ~ CLASS_FRAME + phase_num + C(lang3) + log_len", df).fit(disp=0)
    m1 = smf.logit(f"{y} ~ CLASS_FRAME * phase_num + C(lang3) + log_len", df).fit(disp=0)
    lr = 2 * (m1.llf - m0.llf)
    rows.append({"outcome": y, "LR": lr, "p": stats.chi2.sf(lr, 1),
                 "interaction_OR": np.exp(m1.params["CLASS_FRAME:phase_num"])})
inter = pd.DataFrame(rows).set_index("outcome")
inter["p_holm"] = multipletests(inter.p, method="holm")[1]
inter.round(3)

c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,LR,p,interaction_OR,p_holm
outcome,,,,
NEG_EMOTION,16.227,0.000,1.513,0.000
ANGER,16.659,0.000,1.527,0.000
CONTEMPT,1.227,0.268,1.238,0.804
SADNESS,0.016,0.899,0.982,0.899
COMPASSION_VICTIMS,0.747,0.388,0.773,0.804
BROAD_OUTRAGE,10.636,0.001,1.507,0.004


## Robustness - rerun the class vs emotion models on stricter subsets

In [19]:
df.autofix.value_counts(dropna=False)

autofix
NaN                                                                                   9166
language_check:no_filipino_words                                                      1200
stance->NEUTRAL(needs_context)                                                         642
reject_reasons->NA                                                                     299
language_check:filipino_words_in_english                                               129
language_check:no_filipino_words;stance->NEUTRAL(needs_context)                         37
stance->NEUTRAL(needs_context);reject_reasons->NA                                       30
language_check:no_filipino_words;reject_reasons->NA                                     29
language_check:filipino_words_in_english;stance->NEUTRAL(needs_context)                  6
class_direction->NA                                                                      4
language_check:filipino_words_in_english;reject_reasons->NA                       

In [20]:
subsets = {"all": df}
if "confidence" in df:
    subsets["conf HIGH/MEDIUM"] = df[df.confidence.astype(str).str.upper().isin(["HIGH", "MEDIUM"])]
if "autofix" in df:
    subsets["autofix False"] = df[df.autofix.astype(str).str.lower().isin(["false", "0"])]
subsets["no NEEDS_CONTEXT"] = df[df.NEEDS_CONTEXT == 0]

for k, d in subsets.items():
    print(k, len(d), "class-framed:", int(d.CLASS_FRAME.sum()))

subsets = {k: d for k, d in subsets.items() if d.CLASS_FRAME.sum() > 0}
rob = pd.concat({k: emotion_table(d)[["OR", "p_holm"]] for k, d in subsets.items()}, axis=1)
rob.round(3)

all 11551 class-framed: 414
conf HIGH/MEDIUM 10102 class-framed: 414
autofix False 0 class-framed: 0
no NEEDS_CONTEXT 9542 class-framed: 349


c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\PHSP8131\crisis-management-analysis\.venv\Lib\site-packages\statsmodels\discrete\di

all        conf HIGH/MEDIUM        no NEEDS_CONTEXT  \
                       OR p_holm               OR p_holm               OR   
outcome                                                                     
NEG_EMOTION         0.567  0.021            0.564  0.018            0.549   
ANY_EMOTION         0.325  0.000            0.311  0.000            0.290   
BROAD_OUTRAGE       0.676  0.069            0.674  0.065            0.603   
ANGER               0.633  0.087            0.630  0.079            0.611   
CONTEMPT            0.410  0.000            0.409  0.000            0.365   
DISGUST             1.807  0.415            1.806  0.415            1.735   
SADNESS             0.379  0.000            0.366  0.000            0.369   
CONDOLENCE          0.309  0.000            0.301  0.000            0.307   
COMPASSION_VICTIMS  0.164  0.000            0.159  0.000            0.124   
SYMPATHY_OFFENDER   0.000  0.000            0.000  0.000            0.000   
FEAR                0.376  0.415            0.376  0.415            0.481   

                           
                   p_holm  
outcome                    
NEG_EMOTION         0.017  
ANY_EMOTION         0.000  
BROAD_OUTRAGE       0.059  
ANGER               0.073  
CONTEMPT            0.000  
DISGUST             0.609  
SADNESS             0.000  
CONDOLENCE          0.000  
COMPASSION_VICTIMS  0.000  
SYMPATHY_OFFENDER   0.000  
FEAR                0.609

## Qualitative anchors - random class-framed comments per phase and direction

In [21]:
pd.set_option("display.max_colwidth", 300)
ex = (cf.sample(frac=1, random_state=1)
        .groupby(["phase", "dir3"], observed=True)
        .head(2)
        .sort_values(["phase", "dir3"]))
ex[["phase", "dir3", "comment_id", "text"]]

,phase,dir3,comment_id,text
491,01-onset,INEQUALITY_JUSTICE,Ugwx_bsXjkRIgYTdW7t4AaABAg,kawawa naman ang resort ow er mukang sila ang fall guy ng drowning incident mayaman kase ang ateneo kaya di kaya ituro na rezponsible ng mga im estigador pera na naman ang iiral dyan
827,01-onset,INEQUALITY_JUSTICE,UgyZsumLaYNyZaX_nKt4AaABAg.AXnUx6awfHmAYNHNamssiY,user_6d92f166 kya nga. mhirap laban sa mayaman. sana nag lasalle na lang sila at tlgang masama ugali ng mga taga ateneo devil killer de manila. kiefer ahas at sam guilt reyes
233,01-onset,VICTIMS_POOR,Ugxhim8db3TEec0y0y94AaABAg,nakakaiyak pero ngayon na nakikita ko yung mga abogado nang pamilya parang politika na eh puro mga dds lawyers for what? magpapabango kasi taga davao.
245,01-onset,VICTIMS_POOR,Ugx08lB9z8BmMoz-Fr14AaABAg,akala ko ba matalino ang ateneo? bakit hindi nila inisip ang safety? dapat sila ang itapon sa dagat na may 20kilos sa paa at ang coach..
3177,02-confirmation,ELITE_OR_OTHER,Ugypx59mTOiMbFJce4p4AaABAg,"for sure, there's gonna be a cover-up on this incident. its an elite university, that's all."
2733,02-confirmation,ELITE_OR_OTHER,Ugxank1lfgvfisUHQrx4AaABAg.AXyn3X7KtyqAXzZwmTgeCp,patawa ka din ale! ano kinalaman ng government eh private resort nga yun at yung school din private school! wag shunga-shunga makadamay lng sa government!
1357,02-confirmation,INEQUALITY_JUSTICE,Ugxi3mwoVjN6rvW28Nh4AaABAg,"the way this police answered clearly shows there was no thorough investigation..hopefully this won’t go away, justice will prevail knowing philippines justice system always sides with people with money."
2573,02-confirmation,INEQUALITY_JUSTICE,UgxkIyr_shVOGgLUKRp4AaABAg,e mandatory na lahat ng resorts meron available and.on hand at ready equipped na 3 to 4 units na rescue :ring_buoy: heavy duty jet skis tulad da usa and australia
2997,02-confirmation,VICTIMS_POOR,UgwoTxfnyZxKqI4zbW54AaABAg,"dapat hanggat estudyante, kailangan ng consents ng magulang kahit na 18 and above na"
2848,02-confirmation,VICTIMS_POOR,Ugxsk08yjcQRJkYlpPF4AaABAg,asan na yong mga nagmamahal kay coach tab. puro kayo bring back coach baldwin dati


## Validation sample for hand coding (blind) - use for Cohen's kappa

In [22]:
rng = 42
pos = df[df.CLASS_FRAME == 1].sample(75, random_state=rng)
neg = df[df.CLASS_FRAME == 0].sample(75, random_state=rng)
samp = pd.concat([pos, neg]).sample(frac=1, random_state=rng)

blind = samp[["comment_id", "phase", "text"]].copy()
for c in ["hand_CLASS_FRAME", "hand_CLASS_DIRECTION", "hand_EMOTIONS"]:
    blind[c] = ""
blind.to_csv(OUT / "validation_sample_blind.csv", index=False)
samp[["comment_id", "CLASS_FRAME", "CLASS_DIRECTION"] + EMO[3:]].to_csv(OUT / "validation_sample_llm_key.csv", index=False)
print(len(blind))

150


After hand coding, compute agreement:
`from sklearn.metrics import cohen_kappa_score` then `cohen_kappa_score(hand, llm)` per variable.
The sample is stratified (half class-framed), so also report precision and recall for CLASS_FRAME, not kappa alone.